# Using Indicators (pandas only)

The `mintalib.indicators` module provides composable indicator objects that bind a calculation with its parameters (pandas only — for polars, use `mintalib.expressions`).

Indicators are named in **upper case** (e.g. `SMA`, `EMA`, `MACD`). An indicator instance is callable and can be passed directly to `prices.assign()` or invoked as `SMA(50)(prices)`. The `|` operator chains indicators: `EMA(20) | ROC(1)` means ROC applied after EMA.

In [1]:
import numpy as np
import pandas as pd

from mintalib.samples import sample_prices
from mintalib.indicators import EMA, SMA, ROC, RSI, EVAL, LOG, BBANDS

## Basic Usage

An indicator instance is a callable. Applied to a DataFrame, series-based indicators use the `close` column by default — the `item` parameter selects another column. A pandas Series or numpy array can be passed directly as well (results always come back as pandas objects):

In [2]:
prices = sample_prices()

SMA(50)(prices)

date
1980-12-12           NaN
1980-12-15           NaN
1980-12-16           NaN
1980-12-17           NaN
1980-12-18           NaN
                 ...    
2026-07-31    309.499400
2026-08-03    309.522800
2026-08-04    309.610601
2026-08-05    309.654200
2026-08-06    309.772801
Length: 11504, dtype: float64

In [3]:
SMA(50, item="open")(prices)

date
1980-12-12         NaN
1980-12-15         NaN
1980-12-16         NaN
1980-12-17         NaN
1980-12-18         NaN
                ...   
2026-07-31    308.7480
2026-08-03    308.9760
2026-08-04    309.0094
2026-08-05    309.0742
2026-08-06    309.1698
Length: 11504, dtype: float64

In [4]:
RSI(14)(prices["close"])

date
1980-12-12          NaN
1980-12-15          NaN
1980-12-16          NaN
1980-12-17          NaN
1980-12-18          NaN
                ...    
2026-07-31    43.245950
2026-08-03    40.340781
2026-08-04    44.685147
2026-08-05    45.839619
2026-08-06    48.183314
Length: 11504, dtype: float64

## Chaining

The `|` operator chains indicators left to right: `LOG() | EMA(20) | ROC(1)` applies `LOG` first, then `EMA`, then `ROC`. `.then()` is the fluent equivalent, and `.alias()` names the result:

In [5]:
(LOG() | EMA(20) | ROC(1)).alias("trend")(prices)

date
1980-12-12         NaN
1980-12-15         NaN
1980-12-16         NaN
1980-12-17         NaN
1980-12-18         NaN
                ...   
2026-07-31   -0.077130
2026-08-03   -0.099409
2026-08-04   -0.057922
2026-08-05   -0.043810
2026-08-06   -0.022424
Name: trend, Length: 11504, dtype: float64

## The Assign Idiom

Because indicators are callables, they can be passed directly to `prices.assign`, which invokes each with the DataFrame. `EVAL` evaluates a pandas expression string against the columns — and since `assign` processes keyword arguments sequentially, it can reference columns created earlier in the same call:

In [6]:
result = prices.assign(
    sma50 = SMA(50),
    sma200 = SMA(200),
    rsi = RSI(14),
    slope = LOG() | EMA(20) | ROC(1),
    uptrend = EVAL("sma50 > sma200")
).iloc[:, -5:]

result

,sma50,sma200,rsi,slope,uptrend
date,,,,,
1980-12-12,NaN,NaN,NaN,NaN,0.0
1980-12-15,NaN,NaN,NaN,NaN,0.0
1980-12-16,NaN,NaN,NaN,NaN,0.0
1980-12-17,NaN,NaN,NaN,NaN,0.0
1980-12-18,NaN,NaN,NaN,NaN,0.0
...,...,...,...,...,...
2026-07-31,309.499400,277.661274,43.245950,-0.077130,1.0
2026-08-03,309.522800,277.943019,40.340781,-0.099409,1.0
2026-08-04,309.610601,278.246737,44.685147,-0.057922,1.0


## Multi-Output Indicators

Multi-output indicators return a DataFrame, so they cannot be assigned to a single column — join the result instead:

In [7]:
prices.join(BBANDS(20)(prices))

,open,high,low,close,volume,upperband,middleband,lowerband
date,,,,,,,,
1980-12-12,0.098207,0.098634,0.098207,0.098207,469033600,NaN,NaN,NaN
1980-12-15,0.093510,0.093510,0.093083,0.093083,175884800,NaN,NaN,NaN
1980-12-16,0.086678,0.086678,0.086251,0.086251,105728000,NaN,NaN,NaN
1980-12-17,0.088386,0.088813,0.088386,0.088386,86441600,NaN,NaN,NaN
1980-12-18,0.090949,0.091376,0.090949,0.090949,73449600,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2026-07-31,304.809998,310.690002,300.000000,308.910004,132489100,344.002618,324.3670,304.731382
2026-08-03,309.579987,311.799988,302.559998,303.420013,75052000,345.001166,323.9050,302.808834
2026-08-04,302.730011,310.420013,301.320007,309.380005,68001000,345.104607,323.8410,302.577394


## Pandas Expressions

With pandas >= 3.0, `as_expr()` converts an indicator into a pandas `Expression`. For multi-output indicators, `as_expr(item)` picks a single output — which makes them usable inside `assign` after all:

In [8]:
prices.assign(
    upper = BBANDS(20).as_expr("upperband"),
    lower = BBANDS(20).as_expr("lowerband"),
).iloc[:, -2:]

,upper,lower
date,,
1980-12-12,NaN,NaN
1980-12-15,NaN,NaN
1980-12-16,NaN,NaN
1980-12-17,NaN,NaN
1980-12-18,NaN,NaN
...,...,...
2026-07-31,344.002618,304.731382
2026-08-03,345.001166,302.808834
2026-08-04,345.104607,302.577394
